In [1]:
import sys
from pathlib import Path
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\examples')
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as ag
assert ag.__version__ == '2.1.3'
assert 'site-packages' in Path(ag.__file__).parts


# Challenge completo: pedidos

Tools → Skills → varios Agents → System con decisiones; Environment y Eval sobre la misma declaración.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as ag


In [3]:
from order_fulfillment import build_system, verify
import inspect
from IPython.display import Markdown, display
display(Markdown("```python\n" + inspect.getsource(build_system) + "\n```"))
reports = [verify(backend) for backend in ("native", "langgraph")]
assert all(report["cases_passed"] == 7 for report in reports)
for report in reports:
    for case in report["cases"]:
        print(report["backend"], case["name"], case["output"])
assert [c["output"] for c in reports[0]["cases"]] == [c["output"] for c in reports[1]["cases"]]
print("14/14 contratos; Environment y Eval 6 casos por backend; fallo técnico conservado")

```python
def build_system(backend="native"):
    intake = Agent("intake", Sequence((
        Action("normalize", "normalize_order", input=as_state),
        Action("validate", "validate_order", input=as_state),
    )), skills=(skills.intake,), backend=backend)
    pricing = Agent("pricing", Action("price", "price_order", input=as_state),
                    skills=(skills.pricing,), backend=backend)
    inventory = Agent("inventory", Action("stock", "check_inventory", input=as_state),
                      skills=(skills.inventory,), backend=backend)
    policy = Agent("policy", Action("assess", "assess_policy", input=as_state),
                   skills=(skills.policy,), backend=backend)
    approve = Agent("approve", Action("approve", "approve_order", input=as_state),
                    skills=(skills.approval,), backend=backend)
    review = Agent("review", Action("review", "review_order", input=as_state),
                   skills=(skills.review,), backend=backend)
    reject = Agent("reject", Action("reject", "reject_order", input=as_state),
                   skills=(skills.rejection,), backend=backend)
    return System("order_fulfillment", Graph(
        start="intake",
        nodes=(("intake", intake),
               ("checks", Parallel((pricing, inventory), join=merge_checks)),
               ("policy", policy), ("approve", approve),
               ("review", review), ("reject", reject)),
        edges=(("intake", after_intake), ("checks", "policy"),
               ("policy", after_policy), ("approve", "END"),
               ("review", "END"), ("reject", "END")),
    ), backend=backend)

```

native approve {'order_id': 'PO-001', 'decision': 'approved', 'total_cents': 75000, 'currency': 'MXN', 'reasons': [], 'next_action': 'prepare_shipment'}
native budget {'order_id': 'PO-002', 'decision': 'rejected', 'total_cents': 75000, 'currency': 'MXN', 'reasons': ['over_budget'], 'next_action': 'notify_customer'}
native stock {'order_id': 'PO-003', 'decision': 'rejected', 'total_cents': 1005000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}
native review {'order_id': 'PO-004', 'decision': 'review', 'total_cents': 255000, 'currency': 'MXN', 'reasons': ['high_value_order'], 'next_action': 'request_human_approval'}
native invalid {'order_id': 'PO-005', 'decision': 'rejected', 'total_cents': None, 'currency': 'MXN', 'reasons': ['invalid_quantity'], 'next_action': 'notify_customer'}
native duplicate_lines {'order_id': 'PO-006', 'decision': 'rejected', 'total_cents': 732000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'noti

In [4]:
assert blocked_attempts() == 0